# Sudoku Knowledge Representation & Inference

You will implement three functions -- `build_general_kb`, `build_definite_kb`, `pl_bc_entails` -- and the full-grid solving logic in `sudoku_solver.py`.

This notebook imports and tests those functions. The Streamlit app (`sudoku_app.py`) must import the same implementation from `sudoku_solver.py`; do not copy or rewrite the solver functions inside the app.

**Rules:**

- In `sudoku_solver.py`, import only from `utils.py` and `logic_.py`; do not modify either file.
- Do not duplicate the core solver functions in this notebook or `sudoku_app.py`.
- All other content in this notebook may be edited freely.


## Group details and contributions

**Group number:** 23

| Member | Student name | Student ID | Actual contribution |
| --- | --- | --- | --- |
| 1 | [Name] | [ID] | [Describe work actually completed] |
| 2 | [Name] | [ID] | [Describe work actually completed] |
| 3 | [Name] | [ID] | [Describe work actually completed] |
| 4 | [Name] | [ID] | [Describe work actually completed] |

Complete this table before submission. Replace these placeholders with actual contributions after the work is done. Fill student IDs in the local submission copy unless your group intends to publish them in this public repository.


In [1]:
from utils import *
from logic_ import *
import json
import time
import importlib
import sudoku_solver

# Reload so edits made to sudoku_solver.py are picked up when this cell is rerun.
importlib.reload(sudoku_solver)

from sudoku_solver import (
    atom,
    build_general_kb,
    build_definite_kb,
    solve_full_grid_fc,
    pl_bc_entails,
    solve_full_grid_bc,
    solve_full_grid_bc_with_trace,
)


## Loading a puzzle from JSON

Puzzles are provided as JSON, not embedded in this notebook. Each file looks like:

```json
{
  "n": 9, "box_h": 3, "box_w": 3,
  "puzzles": [
    {
      "givens": {"1_1": 3, "2_3": 1, ...},
      "given_count": 28,
      "solution": {"1_1": 3, "1_2": 4, ...}
    },
    ...
  ]
}
```

`"r_c"` string keys map to the value at row `r`, column `c` (1-indexed). `given_count` is exactly how many cells are given. `solution` is included so you can check your own work as you go, but your functions must not read `solution` to answer a query, they should only read `givens`.

In [2]:
#do not change this function, it is used to load the puzzle pool from a json file
def load_pool(path):
    with open(path) as f:
        raw = json.load(f)
    puzzles = []
    for p in raw['puzzles']:
        givens = {tuple(int(x) for x in k.split('_')): v for k, v in p['givens'].items()}
        solution = {tuple(int(x) for x in k.split('_')): v for k, v in p['solution'].items()}
        puzzles.append({'givens': givens, 'solution': solution, 'given_count': p['given_count']})
    return raw['n'], raw['box_h'], raw['box_w'], puzzles

n, box_h, box_w, puzzle_pool = load_pool('puzzles.json')
print(f'{len(puzzle_pool)} puzzles loaded, {n}x{n} grid, {box_h}x{box_w} boxes')
print('given_count values:', sorted(p['given_count'] for p in puzzle_pool))

# Pick one puzzle to work with through the rest of this notebook.
puzzle = puzzle_pool[0]
givens = puzzle['givens']
print(f"working puzzle has {puzzle['given_count']} givens")
for r in range(1, n + 1):
    print([givens.get((r, c), '.') for c in range(1, n + 1)])

5 puzzles loaded, 9x9 grid, 3x3 boxes
given_count values: [30, 33, 36, 39, 42]
working puzzle has 30 givens
['.', 3, '.', '.', '.', '.', '.', '.', '.']
[2, '.', '.', '.', '.', 7, 8, 6, '.']
[5, 8, '.', 2, 6, '.', '.', 3, '.']
[7, 5, '.', '.', '.', '.', '.', 8, '.']
['.', '.', '.', '.', 7, '.', 5, '.', 4]
['.', '.', '.', 5, 3, '.', '.', 9, 6]
['.', 1, 2, '.', '.', 9, '.', '.', '.']
[6, 4, '.', '.', 5, 8, 9, '.', '.']
['.', '.', '.', '.', 2, 3, '.', '.', '.']


Notice: `pl_fc_entails`/`pl_resolution`/`tt_entails` are all given to you, already implemented, in `logic_.py`. The one algorithm you write yourself in this assignment is **backward chaining** (`pl_bc_entails`) -- see Task 2.

## Define Symbols

Two families of propositional symbols, for row $r$, column $c$, value $v$ (all ranging over $1$ to $n$):

| Symbol | Meaning |
|---|---|
| $\mathit{Is}_{rcv}$ | cell $(r,c)$ has value $v$ |
| $\mathit{Not}_{rcv}$ | cell $(r,c)$ does **not** have value $v$ |

For each representation in Task 1, determine which of these two families is actually needed.

Hint: consider what form a definite (Horn) clause must take, and what `PropDefiniteKB.tell()` will accept.

### Helper function

`atom(prefix, r, c, v)`, where prefix can be either `Is` or `Not`, is a naming helper so propositional symbols need not be typed. It is provided for you in `sudoku_solver.py`; do not change it.

In code, $\mathit{Is}_{rcv}$ and $\mathit{Not}_{rcv}$ are written as single-word symbol names, e.g. `Is3_2_4` and `Not3_2_4`: the prefix is followed immediately by `r`, then by `c` and `v` separated by underscores. No separator is needed between the prefix and `r` since `expr()` only requires a symbol name to start with an uppercase letter. So `Is3_2_4` is parsed as one valid symbol.


In [3]:
# atom() is provided in sudoku_solver.py and imported above.


## Part A: Design the Sudoku Solver

### A.1) Knowledge Representation - Build KB

Every well-posed Sudoku puzzle satisfies exactly these conditions:

- Each cell is assigned **at least one** value from $\{1, \dots, n\}$.
- Each cell is assigned **at most one** value from $\{1, \dots, n\}$, i.e., it cannot hold two different values at once.
- No two cells in the same row hold the same value.
- No two cells in the same column hold the same value.
- No two cells in the same box hold the same value.
- The **givens** cells hold their stated values.

Formalize these Sudoku constraints as propositional logic, in **two** representations:

**(a) General clauses -  `build_general_kb`.**

Encode each of the following directly: no restriction here; you may use arbitrary disjunctions of positive or negated literals. Must return a `PropKB`. 

**(b) Definite (Horn) clauses:- `build_definite_kb`.** Must return a `PropDefiniteKB`. Recall a definite clause is a disjunction of literals with exactly one *positive* literal.

Equivalently written as an implication whose conclusion is a single positive literal and whose premises are a conjunction of positive literals: `P1 & P2 & ... & Pk ==> Q`.

`PropDefiniteKB.tell()` will reject anything else.

Both functions take the puzzle's givens as fixed facts.

- **Implement `build_general_kb()` and `build_definite_kb()` in sudoku_solver.py**
- **Rerun the import cell near the top of this notebook after making changes.**
- **Explain your representation in Conceptual Question 1 below.**

In [4]:
# Implement build_general_kb() and build_definite_kb() in sudoku_solver.py.
# Rerun the import cell near the top of this notebook after making changes.


### A.2) Solve the Puzzle

**(a) Resolution and model checking on the general representation.** Using `build_general_kb` and the library's `pl_resolution` / `tt_entails`, try to solve the puzzle, i.e., for each cell, determine which value is entailed. Both methods are sound and complete on the general KB, but the supplied implementations are expected to take an impractically long time on the full grid.

The experiment below begins with one candidate query, `Is1_1_1`, on the working 9×9 puzzle. Each algorithm runs in its own process with a 30-second limit, so both are attempted and the notebook can continue without a manual kernel interrupt. If even this first query does not finish, we stop rather than attempt all remaining cell/value queries.

Explain what makes resolution's cost grow out of control, and separately what makes truth-table checking's cost grow out of control. Use the observations in Conceptual Question 2 below.

In [5]:
# Run each supplied algorithm separately so one timeout cannot prevent the other
# experiment. Each fresh Python process gets 30 seconds, including KB setup.
# The solver and supplied logic library are used unchanged.
import os
import subprocess
import sys

experiment_payload = json.dumps({
    'n': n, 'box_h': box_h, 'box_w': box_w,
    'givens': [[r, c, v] for (r, c), v in givens.items()],
})
experiment_code = r"""
import json, sys, time
from sudoku_solver import atom, build_general_kb
from logic_ import associate, pl_resolution, tt_entails

method = sys.argv[1]
data = json.loads(sys.argv[2])
givens = {(r, c): v for r, c, v in data['givens']}
kb = build_general_kb(data['n'], data['box_h'], data['box_w'], givens)
query = atom('Is', 1, 1, 1)
start = time.perf_counter()
try:
    if method == 'pl_resolution':
        result = pl_resolution(kb, query)
    else:
        result = tt_entails(associate('&', kb.clauses), query)
    outcome = {'status': 'completed', 'entailed': result}
except (RecursionError, MemoryError) as error:
    outcome = {'status': type(error).__name__, 'detail': str(error)}
outcome['inference_seconds'] = time.perf_counter() - start
print(json.dumps(outcome))
"""

general_kb = build_general_kb(n, box_h, box_w, givens)
query = atom('Is', 1, 1, 1)
symbol_count = len(prop_symbols(associate('&', general_kb.clauses)))
clause_count = len(general_kb.clauses)
print(f'Puzzle: {n}x{n}, {len(givens)} givens; query: {query}')
print(f'General KB: {clause_count:,} stored clauses, {symbol_count} symbols')
print(f'Initial resolution pairs, including ~query: {clause_count * (clause_count + 1) // 2:,}')

experiment_results = {}
for method in ('pl_resolution', 'tt_entails'):
    start = time.perf_counter()
    try:
        process = subprocess.run(
            [sys.executable, '-c', experiment_code, method, experiment_payload],
            capture_output=True, text=True, timeout=30,
            env={**os.environ, 'PYTHONHASHSEED': '0'},
        )
        process.check_returncode()
        outcome = json.loads(process.stdout)
    except subprocess.TimeoutExpired:
        # subprocess.run kills and waits for the child; no solver remains running.
        outcome = {'status': 'timeout', 'entailed': None}
    outcome['wall_seconds'] = time.perf_counter() - start
    experiment_results[method] = outcome
    print(f"{method}: {outcome['status']} after {outcome['wall_seconds']:.2f} s; "
          f"entailed={outcome.get('entailed')}")


Puzzle: 9x9, 30 givens; query: Is1_1_1
General KB: 11,775 stored clauses, 729 symbols
Initial resolution pairs, including ~query: 69,331,200


pl_resolution: timeout after 30.20 s; entailed=None


tt_entails: timeout after 30.03 s; entailed=None


### Observation notes

Record what you observed from the resolution/model-checking experiment here. Use these observations when answering Conceptual Question 2 below.


**Setup.** We used the first supplied 9×9 puzzle with 30 givens, 11,775 stored general-KB clauses and 729 distinct $Is$ symbols. The query was `Is1_1_1`, asking whether the initially blank cell (1,1) must contain 1. The supplied reference solution gives 1 there, but the reference solution was not passed to either inference algorithm.

| Supplied algorithm | Process time limit | Observed wall time | Outcome |
| --- | ---: | ---: | --- |
| `pl_resolution` | 30 s | 30.20 s | Timed out; no entailment result |
| `tt_entails` | 30 s | 30.03 s | Timed out; no entailment result |

Wall time includes process startup, KB construction and termination; the small excess over 30 seconds is termination overhead. The two calls ran independently using the unchanged `logic_.py` functions. A timeout is neither a `False` answer nor evidence that the query is not entailed. We could not finish the first query within the allowance, so did not proceed to solving all cells with these methods.

**Resolution.** The implementation adds the negated query and eagerly constructs every unordered pair of clauses before trying any resolvents. Here this is $\binom{11,776}{2}=69,331,200$ pairs in the first iteration alone. Pair construction already uses $O(C^2)$ time and space for $C$ current clauses. Further rounds can generate exponentially many distinct clauses in the number of symbols, making both pair enumeration and storage grow further. This is an analysis of the supplied code, not a claim that we measured how many resolvents it produced before termination. The observed outcome was a timeout, not a recursion error.

**Truth-table checking.** The implementation recursively branches on all 729 symbols and tests the KB only after a complete assignment has been formed; it does not prune a partial assignment when a constraint is already violated. There are $2^{729}\approx2.824\times10^{219}$ assignments. An entailed query requires exhausting this space, while a non-entailed query may return earlier on a counterexample. In the worst case, time is exponential in the symbol count, multiplied by the cost of evaluating the KB. The implementation traverses assignments depth-first rather than storing all $2^{729}$ assignments at once, so the exponential state count primarily explains the runtime, not an exponential memory requirement.

**(b) Forward chaining on the full grid --** Implement `solve_full_grid_fc()` in sudoku_solver.py. Using your above `build_definite_kb` and the library's `pl_fc_entails` (no need to reimplement them), solve the whole puzzle. Find the value that's entailed for every cell. Reconstruct and display the solved grid.

**(c) Backward chaining -- implement it yourself.**
```python
pl_bc_entails(kb, query) -> bool
```
Implement `pl_bc_entails()` in sudoku_solver.py. Start from the query and recursively try to prove each premise of a rule whose conclusion matches the current goal, bottoming out at known facts. Your function must agree with `pl_fc_entails` on every cell/value pair in this puzzle including correctly returning `False` for values that are *not* part of the solution.

**(d) Backward chaining on the full grid --** Implement `solve_full_grid_bc` in sudoku_solver.py. Using your above `build_definite_kb` and your own `pl_bc_entails`, solve the whole puzzle the same way `solve_full_grid_fc` does: for every cell, try each candidate value until `pl_bc_entails` confirms one. Time both `solve_full_grid_fc` and `solve_full_grid_bc` on the same puzzle and compare. Use your measured result where relevant in Conceptual Question 5.


In [6]:
# Implement solve_full_grid_fc(), pl_bc_entails(), and solve_full_grid_bc()
# in sudoku_solver.py. Rerun the import cell near the top after making changes.


### Verifying the algorithms

Uncomment the following block of code to validate your code.

In [7]:
# Verify solve_full_grid_fc against the puzzle's known solution. Then check
# pl_bc_entails directly, for completeness (it finds the correct value) and
# soundness (it never wrongly confirms an incorrect one), before timing
# solve_full_grid_bc against the same solution.
#
import time

t0 = time.time()
solved = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved == puzzle['solution']

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) == True

# Soundness: pl_bc_entails must not also confirm any incorrect value.
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) == False

t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']

print(f"solve_full_grid_fc: {fc_time:.2f}s")
print(f"solve_full_grid_bc: {bc_time:.2f}s")

solve_full_grid_fc: 0.80s
solve_full_grid_bc: 10.53s


In [8]:
# Five-run timing comparison (each run includes KB and index construction).
import statistics

repeats = 5
times = {'FC': [], 'BC': []}
solvers = {'FC': solve_full_grid_fc, 'BC': solve_full_grid_bc}

for run in range(repeats):
    # Alternate the order to reduce bias from which solver runs first.
    order = ('FC', 'BC') if run % 2 == 0 else ('BC', 'FC')
    for method in order:
        if method == 'BC':
            # Earlier entailment checks may have cached this KB's BC index.
            sudoku_solver._index_memo.cache_clear()
        t0 = time.perf_counter()
        result = solvers[method](n, box_h, box_w, givens)
        elapsed = time.perf_counter() - t0
        assert result == puzzle['solution']
        times[method].append(elapsed)
    print(f"Run {run + 1}: FC={times['FC'][-1]:.3f}s, BC={times['BC'][-1]:.3f}s")

fc_mean = statistics.mean(times['FC'])
fc_std = statistics.stdev(times['FC'])
bc_mean = statistics.mean(times['BC'])
bc_std = statistics.stdev(times['BC'])

print(f"solve_full_grid_fc: {fc_mean:.2f} ± {fc_std:.2f}s")
print(f"solve_full_grid_bc: {bc_mean:.2f} ± {bc_std:.2f}s")

Run 1: FC=0.414s, BC=9.199s


Run 2: FC=0.378s, BC=9.481s


Run 3: FC=0.388s, BC=9.158s


Run 4: FC=0.373s, BC=8.815s


Run 5: FC=0.345s, BC=9.215s
solve_full_grid_fc: 0.38 ± 0.03s
solve_full_grid_bc: 9.17 ± 0.24s


In [9]:
# For Debugging

import time

# FC
t0 = time.time()
solved_fc = solve_full_grid_fc(n, box_h, box_w, givens)
fc_time = time.time() - t0
assert solved_fc == puzzle['solution']
print(f"solve_full_grid_fc: {fc_time:6.2f}s")

# BC
sudoku_solver._index_memo.cache_clear()  # Remove BC index pre-calculation effect
t0 = time.time()
solved_bc = solve_full_grid_bc(n, box_h, box_w, givens)
bc_time = time.time() - t0
assert solved_bc == puzzle['solution']
assert solved_bc == solved_fc
print(f"solve_full_grid_bc: {bc_time:6.2f}s")

# BC with trace
# The trace solver reuses proved facts across cells and records only successful deductions.
solved_bc_trace = solve_full_grid_bc_with_trace(n, box_h, box_w, givens)
print(f"Trace steps: {len(solved_bc_trace['steps'])}")

# First 10 steps
for i, step in enumerate(solved_bc_trace['steps'][:10], start=1):
    print(f"{i}. {step['conclusion']}")
    print(f"   premises: {step['premises']}")
    print(f"   reason: {step['reason']}")

print("\nReconstructed grid:")
for r in range(1, n + 1):
    print(' '.join(str(solved_fc[(r, c)]) for c in range(1, n + 1)))

definite_kb = build_definite_kb(n, box_h, box_w, givens)

# Completeness: pl_bc_entails must find the correct value for every cell.
t0 = time.time()
for (r, c), v in puzzle['solution'].items():
    assert pl_bc_entails(definite_kb, atom('Is', r, c, v)) is True
print(f"\ncompleteness: {n * n} queries, all True  ({time.time() - t0:.1f}s)")

# Soundness: pl_bc_entails must not also confirm any incorrect value.
t0 = time.time()
for (r, c), v in puzzle['solution'].items():
    for other_v in range(1, n + 1):
        if other_v != v:
            assert pl_bc_entails(definite_kb, atom('Is', r, c, other_v)) is False
print(f"soundness:    {n * n * (n - 1)} queries, all False ({time.time() - t0:.1f}s)")

solve_full_grid_fc:   0.39s


solve_full_grid_bc:   8.77s


Trace steps: 489
1. Is2_1_2
   premises: []
   reason: given
2. Not1_1_2
   premises: ['Is2_1_2']
   reason: column_elimination
3. Is1_2_3
   premises: []
   reason: given
4. Not1_1_3
   premises: ['Is1_2_3']
   reason: row_elimination
5. Not2_5_2
   premises: ['Is2_1_2']
   reason: row_elimination
6. Is6_5_3
   premises: []
   reason: given
7. Not2_5_3
   premises: ['Is6_5_3']
   reason: column_elimination
8. Is7_2_1
   premises: []
   reason: given
9. Not7_5_1
   premises: ['Is7_2_1']
   reason: row_elimination
10. Is7_3_2
   premises: []
   reason: given

Reconstructed grid:
1 3 6 9 8 5 4 7 2
2 9 4 3 1 7 8 6 5
5 8 7 2 6 4 1 3 9
7 5 1 4 9 6 2 8 3
3 6 9 8 7 2 5 1 4
4 2 8 5 3 1 7 9 6
8 1 2 6 4 9 3 5 7
6 4 3 7 5 8 9 2 1
9 7 5 1 2 3 6 4 8



completeness: 81 queries, all True  (1.0s)


soundness:    648 queries, all False (17.4s)


## Part B: Conceptual Questions

Answer all five questions directly in this notebook. Replace each **Your answer:** placeholder with your own response.


### 1. Detailed Representation Strategy: General vs. Definite (Horn) Encoding

Explain in detail how you formalized the Sudoku puzzle constraints into propositional logic across both Knowledge Base representations:

**(a) General KB Strategy (`build_general_kb`):** Detail how standard Sudoku rules (e.g., at-least-one value per cell, at-most-one value per cell, row/column/box uniqueness) are directly translated into Conjunctive Normal Form (CNF) clauses without structural restrictions.

**(b) Definite KB Strategy (`build_definite_kb`):** Definite/Horn clauses strictly permit at most one positive literal per clause, prohibiting disjunctive constraints like $(Is_{r,c,1} \lor Is_{r,c,2} \lor Is_{r,c,3} \lor ... \lor Is_{r,c,n})$. Explain step-by-step how your encoding deals with this issue.


**Your answer:**

(a) General KB Strategy (build_general_kb). The general knowledge base represents all six Sudoku constraints directly as Conjunctive Normal Form (CNF) clauses. First, the at-least-one constraint requires each of the 81 cells to contain at least one value from 1–9, producing 81 clauses, each containing nine disjuncts. Second, the at-most-one constraint prevents a cell from containing two different values by adding a clause for every pair of values, resulting in 2916 clauses. Third to fifth, row, column, and box uniqueness are enforced by preventing any two cells within the same unit from sharing the same value. Each constraint type produces 2916 clauses, giving 8748 clauses in total. Finally, the given clues are represented as unit clauses specifying the fixed values of clue cells. Thus, the general KB contains approximately 11775 clauses and directly represents the complete Sudoku problem as a satisfiability instance. Although this representation is straightforward and expressive, reasoning through general CNF using methods such as resolution or exhaustive model checking can become computationally expensive because of the large number of clauses and possible assignments.

(b) Definite KB Strategy (build_definite_kb). The definite KB instead represents Sudoku constraints primarily as Horn-style implication rules, avoiding the large at-least-one disjunction used in the general CNF representation. Rather than explicitly stating that every cell must contain one of nine values, the KB uses elimination rules such as Is_r,c,v -> Not_r,c,w for every w != v, meaning that once a value is assigned to a cell, all other values are eliminated. Similar rules enforce row, column, and box uniqueness: if another cell in the same unit contains value v, then v is eliminated as a candidate from the current cell. These rules can be applied through forward chaining to progressively reduce the possible values of each cell. When eight of the nine possible values have been eliminated, a final positive rule such as Not_r,c,1 .... and Not_r,c,8 -> Is_r,c,9 infers the remaining value. In this way, the definite KB replaces large disjunctive clauses with local elimination and deduction rules, making inference more suitable for forward-chaining reasoning while still capturing the key Sudoku constraints.



### 2. Theoretical Completeness vs. Computational Tractability

Model checking and resolution-refutation are sound and complete—they are guaranteed to terminate with a correct answer for any propositional KB. Despite this guarantee, explain whether you would use either as the default algorithm for solving Sudoku puzzles. *(Hint: Consider space/time complexity and state-space growth, and use your observations from the experiment above where relevant.)*


**Your answer:**

Although model checking (tt_entails) and resolution-refutation (pl_resolution) are both sound and complete, I would not use either as the default algorithm for solving Sudoku because their theoretical guarantees do not make them computationally tractable. Model checking exhaustively evaluates all possible truth assignments. With 81 cells and 9 possible values per cell, the general KB contains 729 propositional variables, giving a search space of 2^729=10^219 possible assignments, with each assignment requiring evaluation against thousands of clauses. This makes exhaustive model checking practically infeasible, which was also observed experimentally when tt_entails failed to complete even for a single query. Resolution-refutation is more sophisticated, but it can suffer from severe clause proliferation as it repeatedly resolves pairs of clauses, potentially generating an exponential number of intermediate clauses. In the experiment, pl_resolution on the general KB typically took more than 30 seconds without completing, demonstrating that its theoretical completeness does not translate into practical efficiency for Sudoku. Both methods also treat the problem as a generic propositional logic problem and therefore fail to exploit Sudoku's strong structural properties, such as local constraints, deterministic assignments, and cascading candidate elimination. In contrast, forward and backward chaining over a definite/Horn KB can directly propagate these constraints without enumerating the entire state space. In our latest five-run experiment, forward chaining solved the 30-given puzzle in an average of 0.38 seconds, while backward chaining with memoization and auto-initialization took an average of 9.17 seconds. Forward chaining is faster here because we must solve all 81 cells, so deriving all facts once is more efficient than proving each cell individually. Therefore, while model checking and resolution are valuable as theoretically correct general-purpose inference methods, I would prefer forward chaining as the default for full-grid Sudoku solving because it exploits the structure of the problem, avoids redundant inference, and provides better practical performance when answering many queries.

### 3. Backward Chaining: Design, Pseudocode, and Challenges

Write pseudocode for `pl_bc_entails(kb, query)`, the backward-chaining algorithm you implemented. Show, at a level of detail that reveals the algorithm's structure (not full Python), how the function checks whether the query is already a known fact, finds candidate rules whose conclusion matches the current goal, recursively proves each premise of such a rule, and combines results—both across the premises of one rule and across multiple candidate rules—to reach a single boolean answer.

Then, in your own words, discuss the design challenges you had to work through to make your algorithm both correct and guaranteed to terminate on every puzzle, and explain how your pseudocode addresses them.


**Your answer:**



### 4. Expressive Limits of Horn Logic

Named elimination techniques such as Naked Pairs and X-Wing can, in fact, be encoded as definite clauses, using the same `Is`/`Not` vocabulary as your `build_definite_kb`. Work out how you would encode one of these techniques as definite clauses, and discuss the consequences of doing so.


**Your answer:**



### 5. Data-Driven vs. Goal-Driven Performance

Forward chaining (data-driven) and backward chaining (goal-driven) are both sound and complete for Horn KBs, but their execution runtimes vary depending on the target query.

**(a)** Describe a scenario—in terms of total KB size versus the query-relevant subset—where backward chaining is significantly faster than forward chaining.

**(b)** Describe a scenario where backward chaining offers no performance advantage, or performs worse than forward chaining.

Use your measured forward- vs. backward-chaining result where relevant.


**(a) When backward chaining can be significantly faster.** Suppose a large Horn KB contains many independent groups of rules, but one requested goal depends on only a few facts and a short chain of rules. With a conclusion index already available, BC follows only rules that can establish that goal and stops as soon as it has a proof. For example, a KB might contain thousands of unrelated rules alongside facts and rules $A$, $A\Rightarrow B$, and $B\Rightarrow G$. To prove $G$, BC only needs that short chain. FC may process many irrelevant facts and fire unrelated rules before reaching $G$, depending on its agenda order; a full-closure FC run also computes all unrelated consequences. The advantage is largest when the query-relevant subset is much smaller than the total KB and there are few queries. Building the KB and index still costs time, so this advantage is most visible when that setup is reused.

**(b) When backward chaining offers no advantage or is slower.** If most of the KB is relevant, or we ask many goals with overlapping dependencies, BC can repeatedly traverse the same rules and re-prove the same subgoals. Failed candidate queries and cyclic dependencies can add more search. FC can instead propagate the known facts once, retain the resulting consequences, and answer many queries by lookup. BC can reduce this disadvantage by sharing successful proofs across queries, so the outcome depends on implementation and workload as well as inference direction.

Our full-grid Sudoku experiment is an example of this second case. We timed the first supplied puzzle (9×9, 30 givens) five times, alternating FC/BC order. Each timed call included KB and index construction. We cleared the BC index cache before each timed BC call, and checked every returned grid against the reference solution. Timings exclude UI rendering and proof-trace generation.

| Solver | Mean time | Sample standard deviation | Runs |
| --- | ---: | ---: | ---: |
| `solve_full_grid_fc` | 0.38 s | 0.03 s | 5 |
| `solve_full_grid_bc` | 9.17 s | 0.24 s | 5 |

FC was approximately 24.1 times faster in this local run. The current FC solver uses one indexed forward-chaining pass to compute all consequences. To exhaust the agenda, it calls the supplied `pl_fc_entails` with a probe absent from the KB, records the processed facts, then reads off the cell values. By contrast, the plain BC solver tries values 1–9 in order for every cell until one is proved. On this completed grid that means 405 entailment calls ($9(1+\cdots+9)$). It reuses the clause index, but each `pl_bc_entails` call starts with a fresh set of proved subgoals, so overlapping deductions are repeated. These costs explain why solving the entire grid favors our FC implementation.

The app's separate `solve_full_grid_bc_with_trace` helper shares proved facts across cells, so its displayed timing need not match the plain BC solver benchmark above. The measurements support FC for this full-grid workload; they do not contradict BC's potential advantage for a small, isolated query.

## Part C: Streamlit Integration

Wrap your Sudoku solver and inference logic into an interactive Streamlit application, `sudoku_app.py`. Your application must implement the following:

**1. Puzzle selection & visual board display**
- An interactive selector/dropdown to pick any puzzle from `puzzles.json`.
- A visual rendering of the grid that clearly distinguishes the initial *givens* from empty cells.

**2. Full-grid auto-solver, with algorithm selection**
- A control (e.g. radio buttons) letting the user choose forward chaining (`solve_full_grid_fc`) or backward chaining (`solve_full_grid_bc`) before solving.
- A button that solves the full grid with the chosen algorithm, renders the solved state, and displays how long the solve took -- so the timing gap from task (d) is visible in the app, not just in the notebook.

**3. Targeted cell entailment query**
- Inputs for row ($r$), column ($c$), and value ($v$).
- A button that checks whether $\mathit{Is}_{rcv}$ is entailed (using `pl_bc_entails`) and displays the boolean verdict (`True` / `False`).

**4. Reasoning trace ("tutor mode")**
- Instrument your chosen inference algorithm (forward or backward chaining) to record the reasoning steps it takes while answering a query.
- Present that trace in a human-readable format -- not a raw Python string or internal symbol dictionary. For example: expandable cards/accordions showing the rule-firing sequence (*"Inferred $\mathit{Not}_{1,2,3}$ because row 1 already contains value 3"* $\implies$ *"Deduce $\mathit{Is}_{1,2,4}$ as the last remaining candidate"*), or plain-English sentences explaining each elimination by row, column, or box constraint.

Import `atom`, `build_definite_kb`, `build_general_kb`, `solve_full_grid_fc`, `solve_full_grid_bc`, and `pl_bc_entails` from `sudoku_solver.py`. Do not copy or rewrite these core functions in the Streamlit file. You may add app-specific helper functions where needed for the interface or reasoning trace.

Refer to the provided `StreamlitDeploymentGuide.pdf` guide to deploy your code and include the link for your Streamlit app below.


## Submission

Submit only the following three files:

1. `Sudoku_Assignment.ipynb` — with all required cells run, outputs visible, and all conceptual questions answered.
2. `sudoku_solver.py` — containing your implementation of the knowledge-base and inference functions.
3. `sudoku_app.py` — containing your Streamlit application.

The provided support files are required to run the assignment but are not part of the student submission.

### Deployed Streamlit app URL

[Group 23 - Sudoku Solver](https://it5005-group23-sudoku.streamlit.app)
